# 🛡️ AI-Powered Predictive Crowd Panic Detection & Containment System
**Team:** Samruddhi D · Priyam Parashar · Meghana D Hegde · Saloni Jadhav  
**Guide:** Prof. Mithun T P | RV College of Engineering

---
### Architecture
1. Dataset Loading (Kaggle API)
2. Video Preprocessing → Frame Extraction
3. Audio Preprocessing → MFCC Features
4. Video Model (MobileNetV2 + LSTM)
5. Audio Model (1D CNN)
6. Evaluation (Accuracy, Precision, Recall, F1)
7. Multimodal Late Fusion
8. Model Export (.h5)
9. Inference Demo


In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q kaggle opencv-python-headless librosa tensorflow scikit-learn matplotlib seaborn tqdm

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────
import os
import cv2
import json
import shutil
import random
import zipfile
import warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f'[INFO] TensorFlow {tf.__version__}')
print(f'[INFO] GPUs available: {len(gpus)} → {gpus}')
DEVICE = '/GPU:0' if gpus else '/CPU:0'
print(f'[INFO] Training on: {DEVICE}')

In [ ]:
# ── Cell 3: Kaggle API Setup ───────────────────────────────────────────────
# Upload kaggle.json when prompted
from google.colab import files

print('Upload your kaggle.json (from https://www.kaggle.com/settings → API → Create New Token)')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('[OK] Kaggle API configured.')

In [ ]:
# ── Cell 4: Download Datasets ──────────────────────────────────────────────
DATA_ROOT = Path('/content/datasets')
DATA_ROOT.mkdir(exist_ok=True)

VIDEO_DS_DIR = DATA_ROOT / 'violence_video'
AUDIO_DS_DIR = DATA_ROOT / 'violence_audio'

VIDEO_DS_DIR.mkdir(exist_ok=True)
AUDIO_DS_DIR.mkdir(exist_ok=True)

# Download Real-Life Violence dataset (primary video dataset)
print('[INFO] Downloading video dataset...')
!kaggle datasets download -d mohamedmustafa/real-life-violence-situations-dataset \
    -p /content/datasets/violence_video --unzip -q

# Download audio dataset
print('[INFO] Downloading audio dataset...')
!kaggle datasets download -d fangfangz/audio-based-violence-detection-dataset \
    -p /content/datasets/violence_audio --unzip -q

print('[OK] Datasets downloaded.')

# List downloaded structure
for p in sorted(DATA_ROOT.rglob('*')):
    if p.is_dir():
        n = len(list(p.iterdir()))
        print(f'  {p.relative_to(DATA_ROOT)} ({n} items)')

In [ ]:
# ── Cell 5: Configuration ──────────────────────────────────────────────────
CFG = {
    # Video
    'SEQ_LEN'     : 16,       # frames per clip
    'IMG_SIZE'    : 112,      # spatial resolution (112 saves memory vs 224)
    'VIDEO_EPOCHS': 15,
    'VIDEO_BS'    : 8,
    'VIDEO_LR'    : 1e-4,

    # Audio
    'SR'          : 22050,
    'DURATION'    : 3.0,      # seconds per sample
    'N_MFCC'      : 40,
    'TIME_STEPS'  : 128,
    'AUDIO_EPOCHS': 20,
    'AUDIO_BS'    : 32,
    'AUDIO_LR'    : 1e-4,

    # Fusion
    'W_VIDEO'     : 0.6,
    'W_AUDIO'     : 0.4,
    'HIGH_THRESH' : 0.70,
    'MED_THRESH'  : 0.40,

    # Paths
    'VIDEO_MODEL' : '/content/models/video_model.h5',
    'AUDIO_MODEL' : '/content/models/audio_model.h5',
}

Path('/content/models').mkdir(exist_ok=True)
print('[CFG]', json.dumps(CFG, indent=2))

In [ ]:
# ── Cell 6: Video Dataset → Build File List ────────────────────────────────
# The Real-Life Violence dataset has:
#   Violence/   → violence clips
#   NonViolence/ → normal clips

def find_video_files(root: Path):
    """Recursively find mp4/avi files and infer labels from parent folder name."""
    records = []
    for ext in ['*.mp4', '*.avi', '*.mov']:
        for path in root.rglob(ext):
            parent = path.parent.name.lower()
            if 'violence' in parent and 'non' not in parent:
                label = 1  # violence
            elif 'non' in parent or 'normal' in parent or 'peaceful' in parent:
                label = 0  # non-violence
            else:
                continue
            records.append({'path': str(path), 'label': label})
    return pd.DataFrame(records)

video_df = find_video_files(VIDEO_DS_DIR)
print(f'[INFO] Video files found: {len(video_df)}')
print(video_df['label'].value_counts().rename({0:'NonViolence', 1:'Violence'}))

# Balance classes
min_count = video_df['label'].value_counts().min()
video_df = (
    video_df.groupby('label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), min_count), random_state=42))
    .reset_index(drop=True)
)
print(f'[INFO] After balancing: {len(video_df)} clips ({min_count} per class)')
video_df.head()

In [ ]:
# ── Cell 7: Frame Extraction ───────────────────────────────────────────────
def extract_frames(video_path: str, n_frames: int = CFG['SEQ_LEN'],
                   img_size: int = CFG['IMG_SIZE']) -> np.ndarray | None:
    """Extract n evenly-spaced frames. Returns (n, H, W, 3) float32 [0,1] or None."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < n_frames:
        cap.release()
        return None

    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            cap.release()
            return None
        frame = cv2.resize(frame, (img_size, img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame.astype(np.float32) / 255.0)
    cap.release()
    return np.stack(frames)  # (N, H, W, 3)


def build_video_dataset(df: pd.DataFrame, max_clips: int = 400):
    """Extract frames for all clips. Returns X (N,SEQ,H,W,3), y (N,) arrays."""
    X, y = [], []
    df = df.sample(min(len(df), max_clips), random_state=42).reset_index(drop=True)
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting frames'):
        frames = extract_frames(row['path'])
        if frames is not None:
            X.append(frames)
            y.append(row['label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


print('[INFO] Building video dataset (this may take a few minutes)...')
X_video, y_video = build_video_dataset(video_df, max_clips=300)
print(f'[OK] X_video shape: {X_video.shape} | y_video shape: {y_video.shape}')
print(f'     Violence: {y_video.sum()} | Non-violence: {(y_video==0).sum()}')

# Visualise sample frames
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for idx, ax in enumerate(axes.flat):
    sample_idx = idx // 8
    frame_idx  = idx % 8 * 2
    ax.imshow(X_video[sample_idx, frame_idx])
    ax.axis('off')
    if frame_idx == 0:
        label_str = 'VIOLENCE' if y_video[sample_idx] == 1 else 'NORMAL'
        ax.set_title(label_str, fontsize=8, color='red' if y_video[sample_idx]==1 else 'green')
plt.suptitle('Sample Video Frames', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/models/sample_frames.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 8: Audio Dataset → MFCC Extraction ───────────────────────────────
def find_audio_files(root: Path):
    records = []
    for ext in ['*.wav', '*.mp3', '*.ogg', '*.flac']:
        for path in root.rglob(ext):
            parent = path.parent.name.lower()
            if 'violence' in parent and 'non' not in parent:
                label = 1
            elif 'non' in parent or 'normal' in parent or 'background' in parent:
                label = 0
            else:
                continue
            records.append({'path': str(path), 'label': label})
    return pd.DataFrame(records)


def extract_mfcc(path: str,
                 sr: int       = CFG['SR'],
                 duration: float = CFG['DURATION'],
                 n_mfcc: int   = CFG['N_MFCC'],
                 time_steps: int = CFG['TIME_STEPS']) -> np.ndarray | None:
    try:
        y, _ = librosa.load(path, sr=sr, duration=duration * 4)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        # Pad / trim to fixed time axis
        if mfcc.shape[1] < time_steps:
            mfcc = np.pad(mfcc, ((0,0),(0, time_steps - mfcc.shape[1])), mode='constant')
        else:
            mfcc = mfcc[:, :time_steps]
        # Z-score normalisation
        mfcc = (mfcc - mfcc.mean()) / (mfcc.std() + 1e-6)
        return mfcc[..., np.newaxis].astype(np.float32)  # (40, 128, 1)
    except Exception as e:
        return None


def build_audio_dataset(df: pd.DataFrame, max_samples: int = 2000):
    X, y = [], []
    df = df.sample(min(len(df), max_samples), random_state=42).reset_index(drop=True)
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting MFCC'):
        feat = extract_mfcc(row['path'])
        if feat is not None:
            X.append(feat)
            y.append(row['label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


audio_df = find_audio_files(AUDIO_DS_DIR)
print(f'[INFO] Audio files: {len(audio_df)}')
print(audio_df['label'].value_counts())

# Balance
min_a = audio_df['label'].value_counts().min()
audio_df = (
    audio_df.groupby('label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), min_a), random_state=42))
    .reset_index(drop=True)
)

print('[INFO] Building audio dataset...')
X_audio, y_audio = build_audio_dataset(audio_df, max_samples=1000)
print(f'[OK] X_audio shape: {X_audio.shape}')

# Visualise MFCCs
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, ax in enumerate(axes):
    img = librosa.display.specshow(X_audio[i,:,:,0], ax=ax, x_axis=None, y_axis=None)
    ax.set_title('Violence' if y_audio[i]==1 else 'Normal',
                 color='red' if y_audio[i]==1 else 'green', fontsize=9)
    ax.axis('off')
plt.suptitle('MFCC Features (40 coefficients × 128 time steps)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/models/sample_mfcc.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 9: Train-Val-Test Splits ──────────────────────────────────────────
# Video split
X_vtr, X_vte, y_vtr, y_vte = train_test_split(
    X_video, y_video, test_size=0.2, stratify=y_video, random_state=42)
X_vtr, X_vval, y_vtr, y_vval = train_test_split(
    X_vtr, y_vtr, test_size=0.15, stratify=y_vtr, random_state=42)

y_vtr_cat  = to_categorical(y_vtr,  2)
y_vval_cat = to_categorical(y_vval, 2)
y_vte_cat  = to_categorical(y_vte,  2)

# Audio split
X_atr, X_ate, y_atr, y_ate = train_test_split(
    X_audio, y_audio, test_size=0.2, stratify=y_audio, random_state=42)
X_atr, X_aval, y_atr, y_aval = train_test_split(
    X_atr, y_atr, test_size=0.15, stratify=y_atr, random_state=42)

y_atr_cat  = to_categorical(y_atr,  2)
y_aval_cat = to_categorical(y_aval, 2)
y_ate_cat  = to_categorical(y_ate,  2)

print(f'Video  → Train: {len(X_vtr)} | Val: {len(X_vval)} | Test: {len(X_vte)}')
print(f'Audio  → Train: {len(X_atr)} | Val: {len(X_aval)} | Test: {len(X_ate)}')

In [ ]:
# ── Cell 10: Build Video Model (MobileNetV2 + LSTM) ────────────────────────
def build_video_model(seq_len=CFG['SEQ_LEN'], img_size=CFG['IMG_SIZE']):
    base = MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet',
    )
    # Fine-tune top 30 layers
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers[-30:]:
        layer.trainable = True

    inp = layers.Input(shape=(seq_len, img_size, img_size, 3), name='video_input')
    x = layers.TimeDistributed(base, name='mobilenet')(inp)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D(), name='gap')(x)
    # Temporal modelling
    x = layers.Bidirectional(layers.LSTM(128, dropout=0.3, recurrent_dropout=0.2), name='bilstm')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu', name='fc2')(x)
    out = layers.Dense(2, activation='softmax', name='output')(x)

    model = Model(inp, out, name='VideoViolenceDetector')
    model.compile(
        optimizer=keras.optimizers.Adam(CFG['VIDEO_LR']),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


with tf.device(DEVICE):
    video_model = build_video_model()

video_model.summary()
print(f'\nTrainable params: {sum(np.prod(v.shape) for v in video_model.trainable_variables):,}')

In [ ]:
# ── Cell 11: Train Video Model ─────────────────────────────────────────────
video_callbacks = [
    callbacks.ModelCheckpoint(
        CFG['VIDEO_MODEL'], monitor='val_accuracy',
        save_best_only=True, verbose=1),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1),
]

with tf.device(DEVICE):
    video_history = video_model.fit(
        X_vtr, y_vtr_cat,
        validation_data=(X_vval, y_vval_cat),
        epochs=CFG['VIDEO_EPOCHS'],
        batch_size=CFG['VIDEO_BS'],
        callbacks=video_callbacks,
        verbose=1,
    )

In [ ]:
# ── Cell 12: Build Audio Model (2D CNN on MFCC) ────────────────────────────
def build_audio_model(n_mfcc=CFG['N_MFCC'], time_steps=CFG['TIME_STEPS']):
    inp = layers.Input(shape=(n_mfcc, time_steps, 1), name='mfcc_input')

    # Block 1
    x = layers.Conv2D(32, (3,3), padding='same', name='conv1')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 2
    x = layers.Conv2D(64, (3,3), padding='same', name='conv2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 3
    x = layers.Conv2D(128, (3,3), padding='same', name='conv3')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 4
    x = layers.Conv2D(256, (3,3), padding='same', name='conv4')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu', name='fc2')(x)
    out = layers.Dense(2, activation='softmax', name='output')(x)

    model = Model(inp, out, name='AudioPanicDetector')
    model.compile(
        optimizer=keras.optimizers.Adam(CFG['AUDIO_LR']),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


with tf.device(DEVICE):
    audio_model = build_audio_model()

audio_model.summary()

In [ ]:
# ── Cell 13: Train Audio Model ─────────────────────────────────────────────
audio_callbacks = [
    callbacks.ModelCheckpoint(
        CFG['AUDIO_MODEL'], monitor='val_accuracy',
        save_best_only=True, verbose=1),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=6,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1),
]

# Data augmentation for audio (time masking simulation)
def augment_mfcc(X):
    X_aug = X.copy()
    for i in range(len(X_aug)):
        if random.random() > 0.5:
            t = random.randint(0, 15)
            t0 = random.randint(0, CFG['TIME_STEPS'] - t)
            X_aug[i, :, t0:t0+t, :] = 0
        if random.random() > 0.5:
            f = random.randint(0, 8)
            f0 = random.randint(0, CFG['N_MFCC'] - f)
            X_aug[i, f0:f0+f, :, :] = 0
    return X_aug

X_atr_aug = augment_mfcc(X_atr)

with tf.device(DEVICE):
    audio_history = audio_model.fit(
        X_atr_aug, y_atr_cat,
        validation_data=(X_aval, y_aval_cat),
        epochs=CFG['AUDIO_EPOCHS'],
        batch_size=CFG['AUDIO_BS'],
        callbacks=audio_callbacks,
        verbose=1,
    )

In [ ]:
# ── Cell 14: Training Curves ───────────────────────────────────────────────
def plot_history(hist, title, savepath):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    axes[0].plot(hist.history['accuracy'],     label='Train Acc', color='#00d4ff', lw=2)
    axes[0].plot(hist.history['val_accuracy'], label='Val Acc',   color='#ff2d55', lw=2)
    axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(hist.history['loss'],     label='Train Loss', color='#00d4ff', lw=2)
    axes[1].plot(hist.history['val_loss'], label='Val Loss',   color='#ff2d55', lw=2)
    axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(savepath, dpi=100, bbox_inches='tight')
    plt.show()

plot_history(video_history, 'Video Model Training', '/content/models/video_training.png')
plot_history(audio_history, 'Audio Model Training', '/content/models/audio_training.png')

In [ ]:
# ── Cell 15: Evaluation Metrics ────────────────────────────────────────────
def evaluate_model(model, X_test, y_test, model_name, class_names=['Normal','Violence']):
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')
    f1   = f1_score(y_test, y_pred, average='weighted')

    print(f'\n═══ {model_name} Evaluation ═══')
    print(f'  Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print()
    print(classification_report(y_test, y_pred, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'{model_name} — Confusion Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'/content/models/{model_name.lower().replace(" ","_")}_cm.png',
                dpi=100, bbox_inches='tight')
    plt.show()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}


video_metrics = evaluate_model(video_model, X_vte, y_vte, 'Video Model')
audio_metrics = evaluate_model(audio_model, X_ate, y_ate, 'Audio Model')

In [ ]:
# ── Cell 16: Multimodal Fusion ─────────────────────────────────────────────
def fuse_predictions(video_probs, audio_probs,
                     w_v=CFG['W_VIDEO'], w_a=CFG['W_AUDIO']):
    """Late fusion: weighted average of class probabilities."""
    return w_v * video_probs + w_a * audio_probs


def risk_level(score: float) -> str:
    if score >= CFG['HIGH_THRESH']: return 'HIGH'
    if score >= CFG['MED_THRESH']:  return 'MEDIUM'
    return 'LOW'


# Demo fusion on a batch (using min test set size for alignment)
n_demo = min(len(X_vte), len(X_ate))
v_probs = video_model.predict(X_vte[:n_demo], verbose=0)
a_probs = audio_model.predict(X_ate[:n_demo], verbose=0)
fused   = fuse_predictions(v_probs, a_probs)

y_fused_pred = np.argmax(fused, axis=1)
y_true_demo  = y_vte[:n_demo]  # use video labels as ground truth

fusion_f1 = f1_score(y_true_demo, y_fused_pred, average='weighted')
fusion_acc = accuracy_score(y_true_demo, y_fused_pred)

print('═══ Fusion Model Results ═══')
print(f'  Accuracy : {fusion_acc:.4f}')
print(f'  F1-Score : {fusion_f1:.4f}')

# Risk level distribution
violence_scores = fused[:, 1]
risk_labels = [risk_level(s) for s in violence_scores]
risk_counts = pd.Series(risk_labels).value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
colors = {'LOW': '#00e87a', 'MEDIUM': '#ffb800', 'HIGH': '#ff2d55'}
bars = ax.bar(risk_counts.index,
              risk_counts.values,
              color=[colors.get(l, '#888') for l in risk_counts.index])
ax.set_title('Risk Level Distribution (Fusion Model)', fontweight='bold')
ax.set_ylabel('Count')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('/content/models/risk_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 17: Metrics Summary Comparison Chart ──────────────────────────────
metrics_df = pd.DataFrame({
    'Model'    : ['Video Model', 'Audio Model'],
    'Accuracy' : [video_metrics['accuracy'],  audio_metrics['accuracy']],
    'Precision': [video_metrics['precision'], audio_metrics['precision']],
    'Recall'   : [video_metrics['recall'],    audio_metrics['recall']],
    'F1-Score' : [video_metrics['f1'],        audio_metrics['f1']],
})

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(metrics_df))
width = 0.2
metric_cols = ['Accuracy','Precision','Recall','F1-Score']
pal = ['#00d4ff','#ff2d55','#ffb800','#00e87a']
for i, (col, color) in enumerate(zip(metric_cols, pal)):
    bars = ax.bar(x + i*width, metrics_df[col], width, label=col, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{bar.get_height():.2f}', ha='center', fontsize=8)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(metrics_df['Model'])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/models/metrics_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print(metrics_df.to_string(index=False))

In [ ]:
# ── Cell 18: Save Models ───────────────────────────────────────────────────
video_model.save(CFG['VIDEO_MODEL'])
audio_model.save(CFG['AUDIO_MODEL'])
print(f'[OK] Video model saved → {CFG["VIDEO_MODEL"]}')
print(f'[OK] Audio model saved → {CFG["AUDIO_MODEL"]}')

# Save config alongside models
with open('/content/models/config.json', 'w') as f:
    json.dump(CFG, f, indent=2)
print('[OK] Config saved → /content/models/config.json')

In [ ]:
# ── Cell 19: Full Inference Demo ───────────────────────────────────────────
def run_inference_demo(video_path: str = None, audio_path: str = None):
    """
    Run inference on a video/audio file or synthetic data.
    Returns a full analysis report dict.
    """
    print('\n' + '═'*50)
    print('  CrowdGuard AI — Inference Demo')
    print('═'*50)

    # Video
    if video_path and Path(video_path).exists():
        frames = extract_frames(video_path)
        if frames is None:
            print('[WARN] Could not extract frames; using test sample.')
            frames = X_vte[0]
    else:
        print('[INFO] No video path provided — using test sample.')
        frames = X_vte[0]

    # Audio
    if audio_path and Path(audio_path).exists():
        mfcc = extract_mfcc(audio_path)
        if mfcc is None:
            mfcc = X_ate[0]
    else:
        print('[INFO] No audio path provided — using test sample.')
        mfcc = X_ate[0]

    # Predict
    v_pred = video_model.predict(frames[np.newaxis], verbose=0)[0]
    a_pred = audio_model.predict(mfcc[np.newaxis], verbose=0)[0]
    fused  = fuse_predictions(v_pred, a_pred)

    violence_score = float(v_pred[1])
    panic_score    = float(a_pred[1])
    fused_score    = float(fused[1])
    level          = risk_level(fused_score)
    confidence     = float(np.max(fused))

    report = {
        'violence_score': round(violence_score, 4),
        'panic_score'   : round(panic_score, 4),
        'fused_score'   : round(fused_score, 4),
        'risk_level'    : level,
        'confidence'    : round(confidence, 4),
        'alert'         : level == 'HIGH',
        'fusion_weights': {'video': CFG['W_VIDEO'], 'audio': CFG['W_AUDIO']},
    }

    # Print report
    print(f'\n  Violence Score : {violence_score*100:.1f}%')
    print(f'  Panic Score    : {panic_score*100:.1f}%')
    print(f'  Fused Score    : {fused_score*100:.1f}%')
    print(f'  Confidence     : {confidence*100:.1f}%')
    print(f'  Risk Level     : {level}')
    if report['alert']:
        print('  ⚠️  ALERT TRIGGERED — Immediate intervention required!')
    else:
        print('  ✅  No critical threat detected.')
    print('═'*50)

    # Visualise
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Sample frames
    axes[0].imshow(frames[CFG['SEQ_LEN']//2])
    axes[0].set_title(f'Mid-Frame | Violence: {violence_score*100:.1f}%', fontsize=9)
    axes[0].axis('off')

    # MFCC
    axes[1].imshow(mfcc[:,:,0], aspect='auto', origin='lower', cmap='viridis')
    axes[1].set_title(f'MFCC Features | Panic: {panic_score*100:.1f}%', fontsize=9)
    axes[1].set_xlabel('Time Steps'); axes[1].set_ylabel('MFCC Coefficients')

    # Risk bar
    bars_data = ['Violence', 'Panic', 'Fused']
    scores    = [violence_score, panic_score, fused_score]
    bar_colors = ['#ff2d55' if s >= 0.7 else '#ffb800' if s >= 0.4 else '#00e87a' for s in scores]
    b = axes[2].bar(bars_data, [s*100 for s in scores], color=bar_colors, width=0.5)
    axes[2].set_ylim(0, 100)
    axes[2].set_ylabel('Score (%)')
    axes[2].set_title(f'Risk Assessment — {level}', fontsize=9,
                      color={'HIGH':'#ff2d55','MEDIUM':'#ffb800','LOW':'#00e87a'}[level])
    axes[2].axhline(70, color='#ff2d55', ls='--', alpha=0.5, label='HIGH (70%)')
    axes[2].axhline(40, color='#ffb800', ls='--', alpha=0.5, label='MED  (40%)')
    axes[2].legend(fontsize=7)
    for bar, s in zip(b, scores):
        axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                     f'{s*100:.1f}%', ha='center', fontsize=9)

    plt.suptitle('CrowdGuard AI — Inference Report', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/models/inference_demo.png', dpi=100, bbox_inches='tight')
    plt.show()

    return report


report = run_inference_demo()
print('\n[JSON Report]')
print(json.dumps(report, indent=2))

In [ ]:
# ── Cell 20: Export all outputs ─────────────────────────────────────────────
from google.colab import files as colab_files

# Zip models + plots
with zipfile.ZipFile('/content/crowdguard_models.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in Path('/content/models').rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to('/content'))

print('[OK] All outputs zipped → /content/crowdguard_models.zip')
colab_files.download('/content/crowdguard_models.zip')
print('[OK] Download started.')